# Lab 09: Production-Grade App with Full LangFuse Observability

**Goal:** Build a complete production-ready FastAPI + LangGraph application with comprehensive LangFuse observability.

**What you'll learn:**
- Integrate LangFuse with the Session 11 production FastAPI app
- Add CallbackHandler to LangGraph agent endpoints
- Implement cost tracking per request
- Add user feedback collection endpoints
- Configure health probes with observability metrics
- Test the complete instrumented system

**Prerequisites:**
- Session 11: Production FastAPI + LangGraph app
- Session 12 Labs 01-08: LangFuse fundamentals

**Architecture:**
```
User Request
    ↓
FastAPI Endpoint (/api/support)
    ↓
LangGraph Agent (with CallbackHandler)
    ↓
LangFuse Mock (traces.json)
    ↓
Cost Tracking + Feedback + Health Metrics
```

## Setup

In [ ]:
import os
import shutil
import json
from datetime import datetime
from typing import TypedDict, Annotated, Optional
from operator import add

WORKDIR = "/tmp/prod-lab-12-09"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

print(f"Working directory: {WORKDIR}")

## Step 1: Real LangFuse Client Setup

Connect to LangFuse cloud using credentials from .env file.

In [ ]:
from langfuse import Langfuse

# Initialize real LangFuse client with credentials from .env
langfuse = Langfuse(
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)

# Verify connection
print(f"✓ LangFuse client initialized")
print(f"  Host: {os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")
print(f"  Public Key: {os.getenv('LANGFUSE_PUBLIC_KEY', 'not set')[:20]}...")
print()
print("🌐 All observability data will be sent to LangFuse cloud")
print("   View your traces at: https://cloud.langfuse.com")

## Step 2: Production LangGraph Agent (from Session 11)

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

load_dotenv()

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Agent State
class SupportState(TypedDict):
    request: str
    employee_name: str
    category: str
    worker_output: str
    error: str
    final_response: str
    audit: Annotated[list, add]
    trace_id: Optional[str]


# Agent Nodes
TEMPLATES = {
    "hr": "Please visit the HR portal or email hr@company.com.",
    "tech": "Please create a Jira ticket or contact IT at ext. 5555.",
    "finance": "Please email finance@company.com with details.",
    "general": "Your request has been noted. A team member will respond shortly.",
}


def supervisor(state: SupportState) -> dict:
    """Classify the support request."""
    prompt = f"Classify as: hr, tech, finance, general. One word.\n{state['request']}"
    try:
        response = llm.invoke(prompt)
        cat = response.content.strip().lower()
        if cat not in ["hr", "tech", "finance", "general"]:
            cat = "general"
    except Exception:
        cat = "general"
    return {
        "category": cat,
        "error": "",
        "audit": [f"Supervisor: classified as {cat}"],
    }


def worker(state: SupportState) -> dict:
    """Generate response for the category."""
    prompt = (
        f"You are {state['category']} support.\n"
        f"Request: {state['request']}\n"
        f"Reply helpfully in 2 sentences."
    )
    try:
        response = llm.invoke(prompt)
        return {
            "worker_output": response.content.strip(),
            "error": "",
            "audit": [f"Worker ({state['category']}) responded"],
        }
    except Exception as e:
        return {
            "worker_output": TEMPLATES.get(state["category"], TEMPLATES["general"]),
            "error": str(e),
            "audit": [f"Worker error, used template"],
        }


def finalize(state: SupportState) -> dict:
    """Format final response."""
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {
        "final_response": f"[{state['category'].upper()}] {state['worker_output']}\n— UniGPS Support | {ts}",
        "audit": [f"Finalized at {ts}"],
    }


# Build LangGraph
graph = StateGraph(SupportState)
graph.add_node("supervisor", supervisor)
graph.add_node("worker", worker)
graph.add_node("finalize", finalize)
graph.add_edge(START, "supervisor")
graph.add_edge("supervisor", "worker")
graph.add_edge("worker", "finalize")
graph.add_edge("finalize", END)
agent = graph.compile()

print("✓ LangGraph agent compiled")

## Step 3: Production FastAPI with LangFuse Instrumentation

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from langfuse.langchain import CallbackHandler
import time

app = FastAPI(
    title="UniGPS Support Agent API",
    version="1.0.0",
    description="Production-grade support agent with LangFuse observability",
)

# Request/Response models
class SupportRequest(BaseModel):
    employee_name: str
    request: str
    session_id: Optional[str] = None


class SupportResponse(BaseModel):
    category: str
    response: str
    audit: list[str]
    trace_id: str
    trace_url: str
    latency_ms: int


class FeedbackRequest(BaseModel):
    trace_id: str
    rating: int  # 1-5
    comment: Optional[str] = None


class HealthResponse(BaseModel):
    status: str
    version: str
    agent: str
    observability: str
    langfuse_host: str


@app.get("/health", response_model=HealthResponse)
async def health():
    """Health probe with observability metrics."""
    return HealthResponse(
        status="healthy",
        version="1.0.0",
        agent="ready",
        observability="langfuse-cloud",
        langfuse_host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    )


@app.post("/api/support", response_model=SupportResponse)
def handle_support(req: SupportRequest):
    """Handle support request with full LangFuse observability."""
    start_time = time.time()
    
    # Create LangFuse CallbackHandler
    langfuse_handler = CallbackHandler()
    
    # Update trace with metadata
    langfuse_handler.update_trace(
        name="support_request",
        user_id=req.employee_name,
        session_id=req.session_id or f"session-{int(time.time())}",
        tags=["production", "support", "fastapi"],
        metadata={
            "endpoint": "/api/support",
            "request_preview": req.request[:100],
        }
    )
    
    # Invoke LangGraph agent with LangFuse callback
    result = agent.invoke(
        {
            "request": req.request,
            "employee_name": req.employee_name,
            "category": "",
            "worker_output": "",
            "error": "",
            "final_response": "",
            "audit": [],
            "trace_id": None,
        },
        config={"callbacks": [langfuse_handler]}
    )
    
    # Calculate latency
    latency_ms = int((time.time() - start_time) * 1000)
    
    # Get trace ID from handler
    trace_id = langfuse_handler.last_trace_id
    
    # Flush main langfuse client to ensure data is sent
    langfuse.flush()
    
    # Get trace URL
    trace_url = f"{os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')}/trace/{trace_id}"
    
    return SupportResponse(
        category=result["category"],
        response=result["final_response"],
        audit=result["audit"],
        trace_id=trace_id,
        trace_url=trace_url,
        latency_ms=latency_ms,
    )


@app.post("/api/feedback")
def submit_feedback(feedback: FeedbackRequest):
    """Submit user feedback for a trace."""
    try:
        # Submit score to LangFuse cloud
        langfuse.create_score(
            trace_id=feedback.trace_id,
            name="user_rating",
            value=feedback.rating,
            comment=feedback.comment,
        )
        langfuse.flush()
        return {
            "status": "success",
            "trace_id": feedback.trace_id,
            "message": "Feedback submitted to LangFuse cloud"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to submit feedback: {str(e)}")


print("✓ FastAPI app configured with LangFuse cloud integration")
print("  All traces will be sent to: https://cloud.langfuse.com")
print("  CallbackHandler automatically manages traces")

## Step 4: Test the Production System

In [ ]:
client = TestClient(app)

# Test 1: Health check
print("=== Test 1: Health Check ===")
resp = client.get("/health")
health_data = resp.json()
print(f"Status: {health_data['status']}")
print(f"Agent: {health_data['agent']}")
print(f"Observability: {health_data['observability']}")
print(f"Total requests: {health_data['total_requests']}")
print()

In [ ]:
# Test 2: Support request with observability
print("=== Test 2: Support Request ===")
resp = client.post("/api/support", json={
    "employee_name": "Priya",
    "request": "I need to apply for sick leave",
    "session_id": "test-session-001",
})

data = resp.json()
print(f"Category: {data['category']}")
print(f"Response: {data['response'][:80]}...")
print(f"Trace ID: {data['trace_id']}")
print(f"Trace URL: {data['trace_url']}")
print(f"Latency: {data['latency_ms']}ms")
print(f"Audit trail: {data['audit']}")
print()
print("🌐 View full trace with LLM calls, costs, and performance metrics:")
print(f"   {data['trace_url']}")
print()

trace_id_1 = data['trace_id']

In [ ]:
# Test 3: Another support request (different category)
print("=== Test 3: Tech Support Request ===")
resp = client.post("/api/support", json={
    "employee_name": "Vikram",
    "request": "My VPN keeps disconnecting",
    "session_id": "test-session-002",
})

data = resp.json()
print(f"Category: {data['category']}")
print(f"Response: {data['response'][:80]}...")
print(f"Trace ID: {data['trace_id']}")
print(f"Trace URL: {data['trace_url']}")
print(f"Latency: {data['latency_ms']}ms")
print()
print("🌐 View trace: {data['trace_url']}")
print()

trace_id_2 = data['trace_id']

In [ ]:
# Test 4: Submit user feedback
print("=== Test 4: User Feedback ===")
resp = client.post("/api/feedback", json={
    "trace_id": trace_id_1,
    "rating": 5,
    "comment": "Very helpful response!",
})

feedback_result = resp.json()
print(f"Feedback submitted: {feedback_result['status']}")
print(f"Message: {feedback_result['message']}")
print()

resp = client.post("/api/feedback", json={
    "trace_id": trace_id_2,
    "rating": 4,
    "comment": "Good, but could be more specific.",
})

feedback_result = resp.json()
print(f"Feedback submitted: {feedback_result['status']}")
print(f"Message: {feedback_result['message']}")
print()
print("🌐 View feedback scores in LangFuse dashboard")
print(f"   https://cloud.langfuse.com")
print()

In [ ]:
# Test 5: View Traces in LangFuse Cloud
print("=== Test 5: View Traces in LangFuse Cloud ===")
print()
print("All traces are now available in the LangFuse cloud dashboard!")
print()
print("🌐 View your traces:")
print(f"   https://cloud.langfuse.com")
print()
print("What you'll see in the dashboard:")
print("  ✓ Trace hierarchy (supervisor → worker → finalize)")
print("  ✓ LLM calls with input/output")
print("  ✓ Token usage and costs per generation")
print("  ✓ Latency metrics for each step")
print("  ✓ User feedback scores (ratings & comments)")
print("  ✓ Session grouping and user filtering")
print("  ✓ Metadata and tags for debugging")
print()
print("Navigate to:")
print("  1. Sessions → View traces grouped by session_id")
print("  2. Traces → View individual traces with full details")
print("  3. Users → Filter traces by employee_name (user_id)")
print("  4. Scores → View all user feedback ratings")
print()

In [ ]:
# Test 6: Final health check
print("=== Test 6: Final Health Check ===")
resp = client.get("/health")
health_data = resp.json()
print(f"Status: {health_data['status']}")
print(f"Version: {health_data['version']}")
print(f"Agent: {health_data['agent']}")
print(f"Observability: {health_data['observability']}")
print(f"LangFuse Host: {health_data['langfuse_host']}")
print()
print("✅ All systems operational!")
print()

## Step 5: Explore LangFuse Cloud Dashboard

Open the LangFuse dashboard to view your traces:

**Dashboard URL:** https://cloud.langfuse.com

### Key Features to Explore:

**1. Traces View**
   - See all traces with timestamps, users, and latencies
   - Click on a trace to see the full execution flow
   - View the supervisor → worker → finalize chain
   - Inspect LLM inputs and outputs

**2. Sessions View**
   - Group traces by session_id
   - Track multi-turn conversations
   - See session-level metrics (total cost, avg latency)

**3. Users View**
   - Filter traces by employee_name (user_id)
   - See per-user costs and usage patterns
   - Identify power users and support patterns

**4. Generations View**
   - All LLM calls across all traces
   - Model usage breakdown (llama-3.3-70b-versatile)
   - Token consumption and costs
   - Response times per generation

**5. Scores View**
   - User feedback ratings (1-5 stars)
   - Comments for qualitative analysis
   - Link scores back to specific traces
   - Calculate average satisfaction scores

**6. Cost Analysis**
   - Total cost across all traces
   - Cost per trace, per user, per session
   - Token usage trends
   - Model cost comparison

### Try These Dashboard Actions:

1. **View Trace Details:** Click on any trace to see the full execution tree
2. **Filter by User:** Find all traces for "Priya" or "Vikram"
3. **Check Costs:** View total spend and per-request costs
4. **Analyze Feedback:** See which traces got 5-star ratings
5. **Debug Issues:** Use metadata and tags to find errors
6. **Compare Sessions:** See how different sessions performed

In [ ]:
# Verify traces were sent to LangFuse
print("=== Verification ===")
print()
print("Your production FastAPI application is now fully instrumented with LangFuse!")
print()
print("✅ Traces are being sent to LangFuse cloud")
print("✅ LLM calls are automatically logged via CallbackHandler")
print("✅ Costs are calculated based on token usage")
print("✅ User feedback is collected and linked to traces")
print("✅ Metadata and tags enable powerful filtering")
print()
print("🎯 Next Steps:")
print("  1. Open https://cloud.langfuse.com in your browser")
print("  2. Navigate to 'Traces' to see your support requests")
print("  3. Click on a trace to explore the LangGraph execution")
print("  4. Check 'Scores' to see user feedback ratings")
print("  5. Review 'Sessions' to see multi-request patterns")
print()

In [ ]:
## LangFuse Dashboard Deep Dive

### Understanding Trace Structure

When you view a trace in LangFuse, you'll see:

```
📊 Trace: support_request
├── 👤 User: Priya
├── 🔖 Session: test-session-001
├── 🏷️ Tags: production, support, fastapi
├── ⏱️ Duration: ~2-3 seconds
├── 💰 Cost: ~$0.00002 (model dependent)
│
├── 🤖 Generation 1: supervisor_classify
│   ├── Model: llama-3.3-70b-versatile
│   ├── Input: "Classify as: hr, tech, finance, general. One word.\nI need to apply for sick leave"
│   ├── Output: "hr"
│   ├── Tokens: ~50 input, ~5 output
│   └── Cost: ~$0.000003
│
├── 🤖 Generation 2: worker_respond
│   ├── Model: llama-3.3-70b-versatile
│   ├── Input: "You are hr support.\nRequest: I need to apply for sick leave\nReply helpfully in 2 sentences."
│   ├── Output: "To apply for sick leave, please visit the HR portal..."
│   ├── Tokens: ~150 input, ~80 output
│   └── Cost: ~$0.000015
│
└── ⭐ Score: user_rating
    ├── Value: 5/5
    └── Comment: "Very helpful response!"
```

### Key Metrics to Monitor

**Performance:**
- Trace duration (end-to-end latency)
- Generation latency (per LLM call)
- Queue time and processing time

**Cost:**
- Cost per trace (total)
- Cost per generation (by model)
- Daily/weekly/monthly spend trends

**Quality:**
- User ratings (scores)
- Feedback comments
- Error rates (if any failures)

**Usage:**
- Traces per user
- Traces per session
- Peak usage times

## TODO 1: Analyze Cost Metrics in LangFuse

**Task:** Open the LangFuse dashboard and analyze cost metrics

**Steps:**
1. Navigate to https://cloud.langfuse.com
2. Go to the "Traces" view
3. Review the cost column for your test traces
4. Click on a trace to see per-generation costs

**Questions to answer:**
- What is the total cost across all your traces?
- What is the average cost per support request?
- Which generation (supervisor vs worker) costs more?
- How does cost vary by category (HR vs Tech)?

**LangFuse Features to Use:**
- **Traces view:** See cost per trace
- **Generations view:** See cost per LLM call
- **Filters:** Filter by user, session, or tags
- **Analytics:** View cost trends over time

**Record your findings:**
```python
# Your analysis
total_cost = "___"  # Total cost from LangFuse dashboard
avg_cost_per_request = "___"  # Average cost per trace
most_expensive_category = "___"  # Which category costs most

print(f"Total Cost: ${total_cost}")
print(f"Average Cost per Request: ${avg_cost_per_request}")
print(f"Most Expensive Category: {most_expensive_category}")
```

In [ ]:
# TODO 1: Cost Analysis (complete after viewing LangFuse dashboard)

# Record your findings from LangFuse cloud
total_cost = "___"  # e.g., "0.00004"
avg_cost_per_request = "___"  # e.g., "0.00002"
supervisor_cost = "___"  # Cost for classification step
worker_cost = "___"  # Cost for response generation
most_expensive_step = "___"  # "supervisor" or "worker"

print("=== Cost Analysis from LangFuse Cloud ===")
print(f"Total cost: ${total_cost}")
print(f"Average cost per request: ${avg_cost_per_request}")
print(f"Supervisor cost: ${supervisor_cost}")
print(f"Worker cost: ${worker_cost}")
print(f"Most expensive step: {most_expensive_step}")
print()
print("💡 Insight: Worker responses cost more due to longer outputs")

## TODO 2: Analyze Quality Metrics in LangFuse

**Task:** Review user feedback scores in the LangFuse dashboard

**Steps:**
1. Navigate to the "Scores" view in LangFuse
2. Review the user_rating scores you submitted
3. Click on a score to see the linked trace

**Questions to answer:**
- What is the average user rating across all traces?
- How many traces received feedback?
- What percentage of traces have 4+ star ratings?
- Are there any patterns in low-rated traces?

**LangFuse Features to Use:**
- **Scores view:** See all feedback ratings
- **Trace linking:** Click score to view associated trace
- **Comments:** Read qualitative feedback
- **Filtering:** Find high/low-rated traces

**Record your findings:**
```python
# Your analysis
avg_rating = "___"  # Average rating from 1-5
feedback_coverage = "___"  # Percentage of traces with scores
high_rated_percentage = "___"  # % with 4+ stars

print(f"Average Rating: {avg_rating}/5")
print(f"Feedback Coverage: {feedback_coverage}%")
print(f"High-Rated Traces: {high_rated_percentage}%")
```

In [ ]:
# TODO 2: Quality Analysis (complete after viewing LangFuse dashboard)

# Record your findings from LangFuse cloud
avg_rating = "___"  # e.g., "4.5"
total_scores = "___"  # Number of scores submitted
high_rated_count = "___"  # Number of 4+ star ratings
feedback_coverage_percent = "___"  # % of traces with scores

print("=== Quality Metrics from LangFuse Cloud ===")
print(f"Average rating: {avg_rating}/5")
print(f"Total scores: {total_scores}")
print(f"High-rated (4+): {high_rated_count}")
print(f"Feedback coverage: {feedback_coverage_percent}%")
print()
print("💡 Insight: High ratings indicate good response quality")
print("💡 Action: Follow up on low-rated traces to improve prompts")

## TODO 3: Production Observability Checklist

Verify that your production application meets these observability requirements by checking the LangFuse dashboard:

**Trace Collection:**
- [ ] All support requests create traces in LangFuse cloud
- [ ] Traces include user_id (employee_name)
- [ ] Traces include session_id for grouping
- [ ] Tags are applied (production, support, fastapi)

**LLM Observability:**
- [ ] All LLM calls (supervisor + worker) are logged as generations
- [ ] Input prompts are captured for debugging
- [ ] Output responses are logged
- [ ] Token usage is tracked per generation

**Cost Tracking:**
- [ ] Per-generation costs are calculated
- [ ] Total trace costs are visible in dashboard
- [ ] Can filter/group by user, session, or date
- [ ] Cost trends can be monitored over time

**Quality Monitoring:**
- [ ] User feedback can be submitted via /api/feedback
- [ ] Scores are linked to traces in LangFuse
- [ ] Comments provide qualitative insights
- [ ] Can identify low-rated traces for improvement

**Production Readiness:**
- [ ] Health endpoint reports LangFuse integration status
- [ ] CallbackHandler automatically captures all agent steps
- [ ] Trace URLs returned to clients for support tickets
- [ ] Metadata enables debugging (endpoint, request preview)

**Performance Metrics:**
- [ ] Latency tracked per request (latency_ms in response)
- [ ] Can measure trace duration in LangFuse
- [ ] Can identify slow generations or bottlenecks
- [ ] Session-level performance analysis available

**Verify in LangFuse Dashboard:**
```
✅ Navigate to each section and confirm:
   - Traces: See your 2 test traces (Priya, Vikram)
   - Generations: See 4 total (2 supervisor + 2 worker)
   - Scores: See 2 user ratings (5 stars, 4 stars)
   - Sessions: See 2 sessions (test-session-001, test-session-002)
   - Users: See 2 users (Priya, Vikram)
```

## Summary

This lab integrated production FastAPI + LangGraph with real LangFuse cloud observability!

**From Session 11 (Production):**
- ✅ FastAPI application with async handling
- ✅ LangGraph multi-agent system (supervisor → worker → finalize)
- ✅ Health probes with observability status
- ✅ Production-grade error handling

**From Session 12 (LangFuse Cloud):**
- ✅ Real LangFuse client with cloud credentials
- ✅ CallbackHandler for automatic trace collection
- ✅ Trace creation with user_id, session_id, tags, metadata
- ✅ Automatic generation logging (LLM calls, tokens, costs)
- ✅ User feedback collection via scores API
- ✅ Cloud dashboard for visualization and analysis

**Key Integration Points:**

1. **Automatic Observability:** CallbackHandler captures all LangGraph steps without manual instrumentation
2. **Rich Context:** Every trace includes user, session, tags, and metadata for powerful filtering
3. **Cost Tracking:** Token usage and model pricing automatically calculate costs per generation
4. **User Feedback Loop:** Scores API links ratings to traces for quality monitoring
5. **Trace URLs:** Each request returns a direct link to the LangFuse dashboard
6. **Production Pattern:** trace → invoke agent with callbacks → flush → return trace_url

**What You Can Now Do:**

🌐 **View Traces:** See full execution flow in LangFuse dashboard  
💰 **Monitor Costs:** Track spending per user, session, or time period  
⭐ **Analyze Quality:** Review user feedback and identify issues  
🔍 **Debug Issues:** Use metadata and tags to filter and find problems  
📊 **Performance Tuning:** Identify slow steps and optimize prompts  
📈 **Business Insights:** Understand usage patterns and support trends

**Production-Ready Features:**
- No manual trace creation needed (CallbackHandler handles it)
- Supports distributed tracing across multiple services
- Scales to thousands of requests per day
- Real-time dashboard updates
- Team collaboration (share traces via URLs)
- Historical analysis and trend monitoring

This is a **production-grade AI agent system** with comprehensive cloud-based observability!

## Key Takeaways

**1. LangFuse Cloud Integration**  
Using the real LangFuse client with cloud credentials enables production-grade observability without managing infrastructure. All traces are stored, visualized, and analyzed in the cloud dashboard.

**2. CallbackHandler Pattern**  
The LangChain CallbackHandler automatically captures all LangGraph steps, LLM calls, and metadata. No manual instrumentation needed—just pass it to `agent.invoke(config={"callbacks": [handler]})`.

**3. Trace URLs for Support**  
Returning trace URLs to clients enables powerful support workflows. Users can reference traces in tickets, and support teams can debug with full context.

**4. Cost Visibility**  
Automatic cost calculation based on token usage and model pricing provides real-time spend visibility. Monitor costs per user, session, or time period to optimize budgets.

**5. Feedback-Driven Improvement**  
Linking user scores to traces creates a quality feedback loop. Identify low-rated traces, analyze what went wrong, and improve prompts or agent logic.

**6. Production Best Practices**
- ✅ Use environment variables for credentials (never hardcode)
- ✅ Flush handlers to ensure data is sent (`handler.flush()`)
- ✅ Include rich metadata for debugging (endpoint, preview, etc.)
- ✅ Tag traces for filtering (environment, feature, user segment)
- ✅ Return trace IDs/URLs to clients for support tickets

**Next Steps in Production:**

1. **Scale Testing:** Test with higher request volumes to validate performance
2. **Alerting:** Set up alerts for cost spikes, high latency, or low ratings
3. **A/B Testing:** Use LangFuse to compare prompt variations and measure impact
4. **Team Sharing:** Collaborate with team members via shared LangFuse workspace
5. **Analytics:** Use LangFuse API or export data for custom analysis
6. **Prompt Management:** Store and version prompts in LangFuse for reproducibility

**Migration Path:**
- **Development:** Use MockLangfuse for local testing (no network calls)
- **Staging:** Use LangFuse cloud with staging project
- **Production:** Use LangFuse cloud with production project (separate for isolation)

**Resources:**
- 📚 LangFuse Docs: https://langfuse.com/docs
- 🎓 LangChain + LangFuse: https://langfuse.com/docs/integrations/langchain
- 💬 Community: https://discord.gg/7NXusRtqYU
- 🐛 Issues: https://github.com/langfuse/langfuse/issues

---

**🎉 Congratulations!** You've built a production-ready AI agent with comprehensive cloud observability!